# Clase 13 · Práctica: casos de uso reales y cierre del curso

## Tu proyecto final

Esta es la última libreta del curso "Advanced Deep Learning". No introduce técnicas nuevas: te pide que combines, en un mini-proyecto de cierre, piezas que ya construiste a lo largo de las doce clases anteriores. Completas dos ejercicios que forman un pipeline (flujo de trabajo) completo de clasificación, de principio a fin, con datos tabulares reales.

- **Ejercicio 1.** Construyes, entrenas y evalúas un perceptrón multicapa (MLP, Multilayer Perceptron) en PyTorch para diagnosticar si un tumor de mama es maligno o benigno, con una capa de *dropout* incluida en la arquitectura.
- **Ejercicio 2.** Reutilizas ese modelo para comprobar, con tus propios datos, el efecto real de la regularización: comparas la exactitud en test de la versión con *dropout* frente a la versión sin él.

Cada ejercicio incluye el andamiaje de código con partes marcadas como `# TODO`, que completas tú. Debajo de cada ejercicio encuentras una solución colapsada: intenta resolver el ejercicio por tu cuenta antes de abrirla.

## Requisito previo

Necesitas haber completado el curso "Advanced Deep Learning", en particular la Clase 5 (Optimización I), donde viste el optimizador Adam y la capa `nn.Dropout` como técnica de regularización.

## Cómo ejecutar esta libreta

Ábrela en Google Colab y ejecuta las celdas en orden, de arriba a abajo. No instales nada adicional: `torch` y `scikit-learn` ya vienen preinstalados en Colab. El dataset se carga directamente desde `scikit-learn`, sin descargas externas.

## Ejercicio 1 · Pipeline completo de diagnóstico de cáncer de mama

### El caso

Colaboras con un laboratorio clínico que quiere una segunda opinión automática sobre biopsias de tejido mamario. Cada muestra llega ya caracterizada por 30 medidas numéricas (radio, textura, perímetro, simetría, entre otras) extraídas de una imagen digitalizada del tejido. Tu tarea es entrenar un modelo que, a partir de esas 30 medidas, clasifique la muestra como maligna o benigna, y que informe con qué fiabilidad lo hace antes de que el equipo clínico se plantee usarlo como apoyo.

Usas el conjunto de datos "Breast Cancer Wisconsin", integrado en `scikit-learn`: 569 muestras, 30 características numéricas por muestra y 2 clases.

### Lo que tienes que hacer

La carga de datos, la partición en entrenamiento y prueba, y la normalización ya están escritas y no necesitas tocarlas. Completa dos partes:

1. La arquitectura del modelo en la clase `MLPClasificador`: una capa oculta, una capa de `nn.Dropout` (que aprendiste en la Clase 5 como técnica de regularización) y la capa de salida.
2. El cálculo de las tres métricas finales sobre el conjunto de test: precisión (*precision*), exhaustividad (*recall*) y la métrica F1, usando las funciones de `sklearn.metrics`.

El bucle de entrenamiento, que usa el optimizador Adam (Adaptive Moment Estimation), ya está escrito: no lo modifiques.

### Pistas

- `MLPClasificador` recibe `num_entradas` (30, una por característica), `num_ocultas` (tamaño de la capa oculta) y `p_dropout` (probabilidad de apagar cada neurona durante el entrenamiento).
- `nn.Dropout(p_dropout)` apaga aleatoriamente una fracción `p_dropout` de las activaciones en cada paso de entrenamiento, y se desactiva automáticamente en modo evaluación (`modelo.eval()`).
- La capa de salida produce 2 logits, uno por clase. No apliques `softmax` explícitamente: `nn.CrossEntropyLoss` ya lo incluye internamente.
- No olvides aplicar ReLU (unidad lineal rectificada) después de la primera capa lineal.
- Para las métricas, `precision_score(y_verdadero, y_predicho)`, `recall_score(y_verdadero, y_predicho)` y `f1_score(y_verdadero, y_predicho)` esperan arrays de NumPy (o listas) con las etiquetas reales y las predichas, en ese orden.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

# Fijamos la semilla para que el resultado sea reproducible
torch.manual_seed(42)
np.random.seed(42)

# Usa GPU (unidad de procesamiento gráfico) si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# Cargamos el dataset de diagnóstico de cáncer de mama: 569 muestras,
# 30 características numéricas por muestra, 2 clases (maligno / benigno)
datos = load_breast_cancer()
X, y = datos.data, datos.target
nombres_clases = datos.target_names
print(f"Clases: {list(nombres_clases)}")

# Separamos en train/test de forma estratificada para conservar
# la proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizamos: media 0, desviación típica 1, ajustado solo con train
# para no filtrar información del conjunto de prueba
escalador = StandardScaler()
X_train = escalador.fit_transform(X_train)
X_test = escalador.transform(X_test)

# Convertimos a tensores de PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

print(f"Muestras de entrenamiento: {X_train_t.shape[0]}, de prueba: {X_test_t.shape[0]}")
print(f"Número de características por muestra: {X_train_t.shape[1]}")

In [ ]:
class MLPClasificador(nn.Module):
    def __init__(self, num_entradas, num_ocultas=32, p_dropout=0.3):
        super().__init__()

        # TODO: define la primera capa lineal.
        # Recibe num_entradas características y produce num_ocultas salidas.
        self.fc1 = ...

        # TODO: define una capa de Dropout con probabilidad p_dropout.
        # Apaga aleatoriamente una fracción de las neuronas durante el
        # entrenamiento para mitigar el sobreajuste (Clase 5).
        self.dropout = ...

        # TODO: define la segunda capa lineal, la de salida.
        # Recibe num_ocultas valores y produce 2 salidas (logits): una por
        # cada clase (maligno / benigno).
        self.fc2 = ...

    def forward(self, x):
        # TODO: aplica fc1 y después la no linealidad ReLU
        x = ...

        # TODO: aplica dropout sobre la salida anterior
        x = ...

        # TODO: aplica fc2 para obtener los logits finales (sin activación)
        x = ...

        return x


modelo = MLPClasificador(num_entradas=X_train_t.shape[1])
modelo = modelo.to(device)
print(modelo)

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

num_epocas = 100
modelo.train()
for epoca in range(1, num_epocas + 1):
    optimizador.zero_grad()
    salidas = modelo(X_train_t)
    perdida = criterio(salidas, y_train_t)
    perdida.backward()
    optimizador.step()

    if epoca % 20 == 0:
        print(f"Época {epoca}/{num_epocas} | pérdida train: {perdida.item():.4f}")

In [ ]:
modelo.eval()
with torch.no_grad():
    salidas_test = modelo(X_test_t)
    predicciones = salidas_test.argmax(dim=1).cpu().numpy()

y_test_np = y_test_t.cpu().numpy()

# TODO: calcula la precisión de las predicciones sobre el conjunto de test
# frente a las etiquetas reales, con precision_score
precision = ...

# TODO: calcula la exhaustividad (recall) con recall_score
exhaustividad = ...

# TODO: calcula la métrica F1 con f1_score
f1 = ...

print(f"Precisión:      {precision:.4f}")
print(f"Exhaustividad:  {exhaustividad:.4f}")
print(f"F1:             {f1:.4f}")

<details>
<summary>💡 Ver solución</summary>

```python
class MLPClasificador(nn.Module):
    def __init__(self, num_entradas, num_ocultas=32, p_dropout=0.3):
        super().__init__()

        # Capa oculta: de num_entradas características a num_ocultas neuronas
        self.fc1 = nn.Linear(num_entradas, num_ocultas)

        # Dropout: apaga aleatoriamente una fracción p_dropout de las
        # neuronas en cada paso de entrenamiento (Clase 5)
        self.dropout = nn.Dropout(p_dropout)

        # Capa de salida: 2 logits, uno por clase (maligno / benigno)
        self.fc2 = nn.Linear(num_ocultas, 2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # logits, sin softmax explícito
        return x


modelo = MLPClasificador(num_entradas=X_train_t.shape[1])
modelo = modelo.to(device)
print(modelo)

# --- Cálculo de métricas finales ---
modelo.eval()
with torch.no_grad():
    salidas_test = modelo(X_test_t)
    predicciones = salidas_test.argmax(dim=1).cpu().numpy()

y_test_np = y_test_t.cpu().numpy()

precision = precision_score(y_test_np, predicciones)
exhaustividad = recall_score(y_test_np, predicciones)
f1 = f1_score(y_test_np, predicciones)

print(f"Precisión:      {precision:.4f}")
print(f"Exhaustividad:  {exhaustividad:.4f}")
print(f"F1:             {f1:.4f}")
```
</details>

## Ejercicio 2 · Comprueba el efecto del *dropout* con tus propios datos

### El caso

El equipo clínico del Ejercicio 1 te pregunta algo muy razonable: "¿de verdad hace falta esa capa de *dropout*? ¿Cuánto cambia el resultado si la quitas?" En la Clase 5 viste, en teoría, que el *dropout* actúa como técnica de regularización y ayuda a que el modelo generalice mejor a datos que no ha visto. Ahora lo compruebas de forma empírica sobre este mismo dataset y este mismo modelo, en lugar de darlo por hecho.

### Lo que tienes que hacer

Completa la función `comparar_regularizacion(X_train, y_train, X_test, y_test)`. Reutiliza la clase `MLPClasificador` del Ejercicio 1 (ya está disponible porque ejecutaste esa celda) para entrenar dos versiones del modelo desde cero, en igualdad de condiciones, y comparar su exactitud sobre el conjunto de test:

1. Un modelo **con** *dropout* activo (`p_dropout=0.3`, el valor por defecto).
2. Un modelo **sin** *dropout* (`p_dropout=0.0`, que equivale a desactivarlo).

Al final, imprime la exactitud de cada versión y una conclusión de una línea sobre cuál generaliza mejor en este conjunto de test concreto.

### Pistas

- Fija la misma semilla (`torch.manual_seed(0)`) justo antes de instanciar cada modelo, para que ambos partan de pesos iniciales comparables y la diferencia se deba solo al *dropout*.
- `nn.Dropout(0.0)` es una capa válida que no apaga ninguna neurona: es una forma sencilla de "desactivar" el *dropout* sin cambiar la arquitectura.
- El resto de la función (el bucle de entrenamiento y el cálculo de exactitud sobre test) ya está resuelto: solo te falta instanciar los dos modelos y escribir la comparación final.
- Con un dataset de solo 569 muestras y una capa oculta pequeña, la diferencia entre ambas versiones puede ser modesta. Eso también es un resultado válido: no todos los modelos pequeños sobreajustan lo suficiente como para que el *dropout* marque una diferencia grande.

In [ ]:
def comparar_regularizacion(X_train, y_train, X_test, y_test, num_epocas=100):
    """Entrena dos versiones del mismo MLP -una con dropout activo y otra
    sin dropout- sobre los mismos datos, y compara su exactitud en test."""

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

    torch.manual_seed(0)
    # TODO: instancia un modelo CON dropout.
    # Usa el valor por defecto de MLPClasificador (p_dropout=0.3) y
    # muévelo al device con .to(device)
    modelo_con_dropout = ...

    torch.manual_seed(0)
    # TODO: instancia un modelo SIN dropout.
    # Pasa p_dropout=0.0 y muévelo también al device con .to(device)
    modelo_sin_dropout = ...

    resultados = {}
    versiones = [
        ("con dropout", modelo_con_dropout),
        ("sin dropout", modelo_sin_dropout),
    ]
    for nombre, modelo_actual in versiones:
        criterio = nn.CrossEntropyLoss()
        optimizador = torch.optim.Adam(modelo_actual.parameters(), lr=1e-3)

        modelo_actual.train()
        for _ in range(num_epocas):
            optimizador.zero_grad()
            salidas = modelo_actual(X_train_t)
            perdida = criterio(salidas, y_train_t)
            perdida.backward()
            optimizador.step()

        modelo_actual.eval()
        with torch.no_grad():
            predicciones = modelo_actual(X_test_t).argmax(dim=1).cpu().numpy()
        exactitud = (predicciones == y_test).mean()
        resultados[nombre] = exactitud

    # TODO: imprime la exactitud de cada versión (recorre `resultados`) y
    # una conclusión de una línea indicando cuál de las dos obtuvo mayor
    # exactitud en este conjunto de test
    ...

    return resultados


resultados = comparar_regularizacion(X_train, y_train, X_test, y_test)

<details>
<summary>💡 Ver solución</summary>

```python
def comparar_regularizacion(X_train, y_train, X_test, y_test, num_epocas=100):
    """Entrena dos versiones del mismo MLP -una con dropout activo y otra
    sin dropout- sobre los mismos datos, y compara su exactitud en test."""

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

    torch.manual_seed(0)
    modelo_con_dropout = MLPClasificador(
        num_entradas=X_train_t.shape[1], p_dropout=0.3
    ).to(device)

    torch.manual_seed(0)
    modelo_sin_dropout = MLPClasificador(
        num_entradas=X_train_t.shape[1], p_dropout=0.0
    ).to(device)

    resultados = {}
    versiones = [
        ("con dropout", modelo_con_dropout),
        ("sin dropout", modelo_sin_dropout),
    ]
    for nombre, modelo_actual in versiones:
        criterio = nn.CrossEntropyLoss()
        optimizador = torch.optim.Adam(modelo_actual.parameters(), lr=1e-3)

        modelo_actual.train()
        for _ in range(num_epocas):
            optimizador.zero_grad()
            salidas = modelo_actual(X_train_t)
            perdida = criterio(salidas, y_train_t)
            perdida.backward()
            optimizador.step()

        modelo_actual.eval()
        with torch.no_grad():
            predicciones = modelo_actual(X_test_t).argmax(dim=1).cpu().numpy()
        exactitud = (predicciones == y_test).mean()
        resultados[nombre] = exactitud

    for nombre, exactitud in resultados.items():
        print(f"Exactitud {nombre}: {exactitud:.4f}")

    mejor = max(resultados, key=resultados.get)
    print(f"\nEn este conjunto de test, la versión '{mejor}' generaliza mejor.")

    return resultados


resultados = comparar_regularizacion(X_train, y_train, X_test, y_test)
```
</details>

## Fin del curso: felicidades

Has llegado al final de "Advanced Deep Learning". Repasa el camino que recorriste:

- Empezaste por los fundamentos: neuronas artificiales, funciones de activación y el algoritmo de retropropagación (*backpropagation*).
- Construiste redes neuronales convolucionales (CNN) para visión por computador y entendiste arquitecturas más profundas y sus bloques residuales.
- Dominaste la optimización: descenso de gradiente, sus variantes, el optimizador Adam, la regularización con *dropout* y otras técnicas para evitar el sobreajuste.
- Trabajaste con secuencias: redes neuronales recurrentes (RNN) y memoria a largo y corto plazo (LSTM, Long Short-Term Memory).
- Entraste en el procesamiento del lenguaje natural (PLN) y en los mecanismos de atención, hasta llegar al Transformer, la arquitectura detrás de los modelos de lenguaje más usados hoy.
- Construiste redes generativas antagónicas (GAN) y exploraste modelos generativos modernos y el aprendizaje no supervisado.
- Cerraste con la inteligencia artificial (IA) responsable, y en esta última clase viste cómo AlphaGo, AlphaFold, la traducción automática, WaveNet, las GAN y los modelos de lenguaje aplican, en producción, exactamente lo que tú acabas de construir en este mini-proyecto: un pipeline completo, con su capa de regularización y su evaluación honesta sobre datos de test.

Cada ejercicio que completaste, en cada una de las trece clases, es una pieza que ya sabes combinar. Ese es el verdadero resultado del curso: no memorizaste fórmulas, entrenaste el criterio para elegir la arquitectura adecuada delante de un problema nuevo.

**Sigue construyendo tus propios proyectos, aplica estas técnicas a un problema real de tu sector y comparte lo que aprendas con tu equipo: ese es el mejor modo de que este conocimiento no se quede en esta libreta.**